# 2b Call Different Models and Routed Resources

**Purpose:**
Use one gateway, one token, and one request format to reach several models, then let a **Routed Resource** pick the model for you.

**What we're doing**

We send the same prompt to every model in your profile and compare the answers. Then we call a Routed Resource, a single model name that the gateway sends to different models based on rules, and watch it choose.

**Why we're doing it**

* In `02a_Use_the_LLM_Gateway.ipynb` you called one model through the gateway. Real applications use several: a small, cheap model for easy requests and a larger one for hard requests.
* Without the gateway, each model would mean another URL, another credential, and another code path. With it, switching models is a one-word change.
* A Routed Resource moves the choice of model out of client code and into a rule your administrator controls

**What to look for**

* The only thing that changes between models is the `"model"` value
* Different models answering the same prompt with different length, quality, and latency
* One alias, two different models answering, depending on the request

**Output**

A side-by-side comparison of the models in your profile, and a Routed Resource sending requests to different models.


## 2.8 How the Gateway Finds a Model

The LLM Gateway has three building blocks. You met the profile in `02a_Use_the_LLM_Gateway.ipynb`; here are all three:

| Object | What it is | Example |
|---|---|---|
| **Resource** | A connection to one upstream model: a provider, a model ID, an endpoint URL, and an API key | Your Section 1 vLLM endpoint, the `gpt-oss-20b` deployment, or an external provider's model |
| **Routed Resource** | A rule set that sends each request to one of several Resources | `smart-router` |
| **Profile** | The models a set of users may call. Each entry points to a Resource or a Routed Resource and gives it an **alias name**. | `frank-test` |

Clients see only the profile's aliases. They never see a Resource's upstream URL or API key:

```
                 X-ClearML-Profile: frank-test
client ── "model": "<alias>" ──▶ LLM Gateway ──▶ Profile ──▶ Resource ─────────────▶ one upstream model
                                                     │
                                                     └────▶ Routed Resource ──rules──▶ Resource A or Resource B
```

> **Note:** Every Resource has its own **Service provider** setting, which tells the gateway how to talk to that upstream model. That is why the client sends the same OpenAI-compatible request to every alias, whatever the provider is.

## 2.9 Load the Token and the Gateway URL, Then Set Your Profile

This is the same setup as `02a_Use_the_LLM_Gateway.ipynb`. First, load the token and the gateway URL from `.env` through `config.py`:

In [18]:
import sys
import time
sys.path.insert(0, "..")
from config import KLIQUE_GATEWAY_TOKEN as token, KLIQUE_LLM_GATEWAY_URL as gateway_url
import requests

print(f"LLM Gateway: {gateway_url}")
print(f"Token:       {token[:8]}...")

LLM Gateway: https://task-router-allegro-master-server.services.internal.allegro.ai/llm
Token:       eyJhbGci...


Then set your profile name, as it is set up in the LLM Gateway. It is not a secret, so set it directly in the cell. Use the same profile as in 02a.

In [19]:
PROFILE = "ServingProfile"  # Your LLM Gateway profile

print(f"Profile: {PROFILE}")

Profile: ServingProfile


The helper below is the `chat()` function from `02a_Use_the_LLM_Gateway.ipynb`, with one addition: it also returns how long the call took, so you can compare models.

Like 02a, it sends `enable_thinking: False` for Qwen3.5 (see Section 1.8.2). Models whose chat template does not use that setting, such as `gpt-oss-20b`, ignore it and still reason before they answer.

In [20]:
def chat(prompt, model_name, max_tokens=256, profile_name=PROFILE, timeout=120):
    """Send a chat completion through the LLM Gateway. Return (parsed reply, seconds taken)."""
    start = time.perf_counter()
    r = requests.post(
        f"{gateway_url}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {token}",
            "X-ClearML-Profile": profile_name,
        },
        json={
            "model": model_name,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "chat_template_kwargs": {"enable_thinking": False},
        },
        timeout=timeout,
    )
    elapsed = time.perf_counter() - start
    if not r.ok:
        # Show the gateway's explanation, e.g. 404 model_not_found.
        print(f"❌ HTTP {r.status_code}: {r.text[:300]}")
    r.raise_for_status()
    return r.json(), elapsed

## 2.10 List the Models in Your Profile

The gateway serves chat requests, not `/v1/models`, so the list of models comes from somewhere else: the Klique API server. Its `routers.get_llm_profiles` call returns every profile you can use. For each profile it gives the aliases it offers, whether each alias is a Routed Resource, and the price, if one is set.

That call takes a different credential from the gateway token: an **access key and secret**. Create them in the WebApp under **Settings > Workspace**. The WebApp shows them with your API server's address. Add all three to `.env`:

```
KLIQUE_API_SERVER=https://<your api server>
KLIQUE_ACCESS_KEY=<your access key>
KLIQUE_SECRET_KEY=<your secret key>
```

The cell below logs in with the key, which returns a short-lived API token, asks for your profiles, and builds `MODELS` from the one named `PROFILE`. Resources go into `MODELS`, and Routed Resources go into `ROUTED`, for later in this notebook.

> **Note:** `routers.get_llm_profiles` is marked *Internal* in the [Klique API reference](https://klique.ai/docs/latest/docs/references/api/routers), so its response may change between releases. The same list is in the WebApp: open your user **Settings**, go to your **Profile** tab, and find **Your AI Endpoints**. If you have no access key, or the call fails, the cell uses `FALLBACK_MODELS` instead. Type your aliases from **Your AI Endpoints** there.

<!-- screenshot: images/your_ai_endpoints.png (Profile tab, Your AI Endpoints listing the profile's aliases) -->

The rest of this notebook expects at least two models in your profile:

| Model | Who runs it |
|---|---|
| Qwen3.5-2B, the model you deployed in Section 1 | You |
| [gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b), OpenAI's 21B-parameter open-weight reasoning model | Your administrator, as a second vLLM Model Deployment instance |

`gpt-oss-20b` runs on Klique exactly the way your Section 1 model does: a vLLM Model Deployment instance with its own Model Endpoint, registered as a Resource in the LLM Gateway. The only difference is size. It is about ten times larger than Qwen3.5-2B, and it reasons before it answers. If the cell lists only one Resource, ask your administrator to add the `gpt-oss-20b` Resource to your profile.

In [21]:
import importlib
import config
importlib.reload(config)  # Re-read .env, in case you added the keys after running 2.9
api_server, access_key, secret_key = config.KLIQUE_API_SERVER, config.KLIQUE_ACCESS_KEY, config.KLIQUE_SECRET_KEY

# Used only if the API call below fails. Copy the aliases from Your AI Endpoints.
FALLBACK_MODELS = {
    "01NotebookModel": "Section 1: Qwen3.5-2B on vLLM, on Klique",
    # "gpt-oss-20b":   "gpt-oss-20b on vLLM, on Klique",
}


def describe(detail):
    """One line for an alias: its price, if the administrator set one."""
    cost = detail.get("cost")
    if not cost:
        return "no price set"
    return f"${cost['in_per_1m']:g} in / ${cost['out_per_1m']:g} out per 1M tokens"


def get_profile(profile_name):
    """Read one LLM Gateway profile from the Klique API server."""
    login = requests.get(f"{api_server}/auth.login", auth=(access_key, secret_key), timeout=30)
    login.raise_for_status()
    api_token = login.json()["data"]["token"]

    r = requests.post(
        f"{api_server}/routers.get_llm_profiles",
        json={},
        headers={"Authorization": f"Bearer {api_token}"},
        timeout=30,
    )
    r.raise_for_status()
    profiles = {p["name"]: p for p in r.json()["data"]["profiles"]}
    if profile_name not in profiles:
        raise KeyError(f"No profile named {profile_name!r}. Profiles you can use: {', '.join(sorted(profiles))}")
    return profiles[profile_name]


try:
    if not (api_server and access_key and secret_key):
        raise RuntimeError("KLIQUE_API_SERVER, KLIQUE_ACCESS_KEY, or KLIQUE_SECRET_KEY is not set in .env")
    profile = get_profile(PROFILE)
    details = profile["model_details"]
    MODELS = {d["name"]: describe(d) for d in details if not d["optimized"]}
    ROUTED = {d["name"]: describe(d) for d in details if d["optimized"]}
    print(f"✅ Read profile {PROFILE!r} from {api_server}")
    print(f"   Default model: {profile['profile_default_model']}\n")
except Exception as e:
    print(f"⚠️ Could not read the profile ({e}). Using FALLBACK_MODELS.\n")
    MODELS, ROUTED = dict(FALLBACK_MODELS), {}

print(f"{'Alias':<28} {'Kind':<16} Price")
for alias, description in MODELS.items():
    print(f"{alias:<28} {'Resource':<16} {description}")
for alias, description in ROUTED.items():
    print(f"{alias:<28} {'Routed Resource':<16} {description}")

✅ Read profile 'ServingProfile' from https://api.allegro-master.hosted.allegro.ai
   Default model: 01NotebookModel

Alias                        Kind             Price
01NotebookModel              Resource         no price set


## 2.11 Same Prompt, Different Models

Now send one prompt to every alias. Look at the request inside `chat()`: the URL, the headers, and the body are identical for every model, except for `"model"`.

In [22]:
prompt = "Explain in two sentences why a team would deploy its own model instead of only using a hosted API."

results = []
for alias, description in MODELS.items():
    print(f"🤖 {alias}  ({description})")
    try:
        reply, seconds = chat(prompt, alias)
    except requests.HTTPError:
        print()
        continue
    answer = (reply["choices"][0]["message"]["content"] or "").strip()
    if not answer:
        answer = "(no answer: the model used all of max_tokens reasoning; see Troubleshooting)"
    tokens = reply["usage"]["completion_tokens"]
    results.append((alias, reply["model"], seconds, tokens))
    print(f"✅ {answer}\n")

print(f"{'Alias':<24} {'Served by':<32} {'Seconds':>8} {'Output tokens':>14}")
for alias, served_by, seconds, tokens in results:
    print(f"{alias:<24} {served_by:<32} {seconds:>8.1f} {tokens:>14}")

🤖 01NotebookModel  (no price set)
❌ HTTP 404: <!DOCTYPE html>
<html>
  <head>
    <meta charset="UTF-8" />
    <title>404 Not Found</title>

    <style>
      * {
        margin: 0;
        padding: 0;
        box-sizing: border-box;
      }
      :root {
        --body-bg-color: #1a1e2c;
        --text-color: #a4adcd;
        --title-size: 10v

Alias                    Served by                         Seconds  Output tokens


Read the table at the bottom:

- **Served by** is the model that actually answered. For your Section 1 alias it is the Model Endpoint Name, as you saw in `02a_Use_the_LLM_Gateway.ipynb`. For other aliases it is the model ID the Resource sends upstream.
- **Seconds** and **Output tokens** show the trade-off. Qwen3.5-2B is small and fast. `gpt-oss-20b` is larger and reasons before it answers, so it usually takes longer and spends more tokens, and in return it gives a more careful answer. Its reasoning tokens count as output tokens, even though they are not in the answer.

> **Checkpoint:** both models answered, and their *Served by* values differ. If only one model ran, your profile has only one Resource; see the end of 2.10. If you typed aliases into `FALLBACK_MODELS`, any alias that printed `❌ HTTP 404` is not in your profile under that name; fix it in 2.10.

Your application did not need a new URL, a new key, or new code for any of these models. That is what the profile buys you. But the application still had to **pick** the alias. The rest of this notebook moves that choice into the gateway.


## 2.12 What a Routed Resource Does

A **Routed Resource** is a single alias that stands for several Resources. When a request arrives, the gateway checks the Routed Resource's rules and forwards the request to the Resource the winning rule names.

Each rule has five fields:

| Field | What it does | Example |
|---|---|---|
| **Priority** | Ranks the rule. Rules are checked from highest priority to lowest, and the first match wins. | `10` |
| **Route on** | The request attribute to test, such as the requested model or the maximum output tokens | max output tokens |
| **Operator** | How to compare the attribute with the value | greater than |
| **Value** | What to compare against | `512` |
| **Route to resource** | Where matching requests go | the `gpt-oss-20b` Resource |

In this lab, the Routed Resource is called **`smart-router`** and uses one idea: *a request that asks for a long answer is a harder request*. Long answers go to `gpt-oss-20b`, and everything else goes to your Section 1 vLLM model.

| Priority | Route on | Operator | Value | Route to resource |
|---|---|---|---|---|
| 10 | max output tokens | greater than | `512` | The `gpt-oss-20b` Resource |
| 1 | max output tokens | less than or equal to | `512` | Your Section 1 vLLM Resource |

The two rules together cover every value of `max_tokens`, so every request matches one of them.

> **Note:** The rules read the request, not the prompt's meaning. A router with these rules does not know whether a question is hard; it knows only what the request asks for. Choose attributes your clients set on purpose.

## 2.13 Set Up the Routed Resource

Routed Resources are created in the LLM Gateway settings, usually by your Klique administrator. If `smart-router` is not already in your profile, give your administrator the two rules from 2.12 and ask them to:

1. Open **Settings → LLM Gateway** and click **Add Routed Resource**.
2. Set **Name** to `smart-router`, and add the two rules from 2.12.
3. Open your profile and click **+ Add Model**. Set **Target type** to **Routed Resource**, choose `smart-router` as the **Target**, and keep `smart-router` as the **Alias name**.

<!-- screenshot: images/add_routed_resource.png (Add Routed Resource dialog with the two rules) -->

<!-- screenshot: images/profile_routed_resource.png (profile's model list with smart-router as a Routed Resource target) -->

Then set the alias in the cell below. If your administrator chose a different alias, use theirs.

In [8]:
ROUTER = "smart-router"  # The Routed Resource's alias in your profile

## 2.14 Watch the Router Choose

Send the same prompt to the same alias twice. The only difference is `max_tokens`: 256, then 1024. Watch **Served by**.

In [23]:
prompt = "List three things to check before putting a model endpoint into production."

for max_tokens in (256, 1024):
    reply, seconds = chat(prompt, ROUTER, max_tokens=max_tokens)
    print(f"max_tokens={max_tokens:<5}  →  served by {reply['model']}  ({seconds:.1f}s)")

NameError: name 'ROUTER' is not defined

The client asked for `smart-router` both times. With `max_tokens=256`, the priority-1 rule matched and your Section 1 model answered. With `max_tokens=1024`, the priority-10 rule matched first and `gpt-oss-20b` answered.

> **Checkpoint:** the two lines name two different models. If both name the same model, check the rules' **Value** and **Operator** with your administrator.

Now read the answers from both models:

In [ ]:
for max_tokens in (256, 1024):
    reply, _ = chat(prompt, ROUTER, max_tokens=max_tokens)
    print(f"--- max_tokens={max_tokens}, served by {reply['model']} ---")
    print((reply["choices"][0]["message"]["content"] or "").strip(), "\n")

## 2.15 Why This Matters

Look at what the client code knows: one URL, one token, one profile, and one model name, `smart-router`. It does not know which models exist behind that name or what the rules are.

That means the routing can change without touching any client:

- **Swap a model.** Point a rule at a new Resource, and every client gets the new model on its next request.
- **Change the threshold.** Raise `512` to `1024`, and more traffic stays on your own GPU.
- **Add a tier.** Add a rule with a higher priority for a third model.

Every one of these is a change in the LLM Gateway, made once, and all of it is metered: open the **AI Usage** dashboard to see the calls from this notebook, split by model.

<!-- screenshot: images/ai_usage_by_model.png (AI Usage dashboard, usage broken down by model) -->


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `404` with `model_not_found` for one alias | That alias is not in your profile, or it is spelled differently. Compare it with **Your AI Endpoints**. |
| `404` for `gpt-oss-20b` | Your administrator has not added the `gpt-oss-20b` Resource to your profile, or gave it a different alias |
| `404` for `smart-router` | The Routed Resource exists but has not been added to your profile, or it has a different alias there |
| Both requests in 2.14 served by the same model | The rule values or operators do not split at the `max_tokens` values you sent, or one rule points at the wrong Resource |
| `502`, `503`, or a timeout from one model | That model's upstream did not answer. For your Section 1 model, check that the instance is still **Active**. |
| No answer from `gpt-oss-20b` in 2.11 | The model used all of `max_tokens` reasoning before it reached the answer. Pass a larger `max_tokens` to `chat()`, for example `chat(prompt, alias, max_tokens=1024)`. |
| `429` | You hit the profile's request rate limit. Wait for the window to pass, then run the cell again. |


## Wrap-Up

You have now used the LLM Gateway three ways:

| Notebook | What you called | Who chose the model |
|---|---|---|
| `02a_Use_the_LLM_Gateway.ipynb` | One alias for one model | You |
| This notebook, 2.11 | Several aliases, one for each model | You, by changing one word |
| This notebook, 2.14 | One Routed Resource alias | The gateway's rules |

**When you are done, stop your Section 1 instance** to release its GPU (Section 1.9).

## Related Docs

- [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [LLM Gateway settings: Resources, Routed Resources, and Profiles](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway)
- [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)